> ⏱ ~25 min

# Round 2 — Grounded

Round 1 measured the bare model. Round 2 gives the same agent a **travel policy binder** and a quick **catalog cheat sheet**, then reruns the same evaluators. The dataset and evaluators do not change — only the instructions do.

**Grounding** is the practice of giving the agent trusted facts to consult while it answers. In our travel scenario, this is the same as putting a printed policy binder on the concierge desk: the concierge can still reason and chat, but the binder keeps recommendations within company rules.

## What you will do

1. Load the travel policy and three catalog summaries.
2. Build a new `Agent` whose instructions include policy and catalog facts.
3. Rerun the same evaluator panel from round 1.
4. Save the metrics to `eval-results/round2-grounded-*` and compare to round 1.

---

In [ ]:
# Suppress experimental/deprecation warnings to keep learning output clean.
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os, sys
from dotenv import load_dotenv

# Locate this lab folder portably (do not hardcode the repo name; learners may fork+rename).
LAB_DIR = Path.cwd().resolve()
if LAB_DIR.name != 'lab-1-foundry-maf':
    for _p in (LAB_DIR, *LAB_DIR.parents):
        _candidate = _p / 'cohere' / 'lab-1-foundry-maf'
        if _candidate.exists():
            LAB_DIR = _candidate
            break
COHERE_DIR = LAB_DIR.parent
load_dotenv(COHERE_DIR / '.env')
sys.path.insert(0, str(LAB_DIR))

PROJECT_ENDPOINT = os.environ['FOUNDRY_PROJECT_ENDPOINT']
DEPLOYMENT = os.getenv('COMMAND_A_DEPLOYMENT', 'command-a')
DATASET = LAB_DIR / '04-eval-dataset.jsonl'
print('Dataset rows:', sum(1 for _ in DATASET.open(encoding='utf-8')))

## 1. Assemble the grounding context

We read the full policy file plus short summaries of each catalog (flights, hotels, cars). Summaries keep token use low while still giving the model concrete record IDs to recommend.

In [ ]:
# Grounding = giving the agent trusted facts (here: policy + catalog summaries)
# without changing the evaluators or the dataset.
import json

policy_text = (LAB_DIR / 'data' / 'travel-policy.md').read_text(encoding='utf-8')

def catalog_summary(name: str) -> str:
    rows = json.loads((LAB_DIR / 'data' / 'catalogs' / name).read_text(encoding='utf-8'))
    return f"{name}: {len(rows)} records; sample ids: " + ', '.join(row['id'] for row in rows[:4])

catalog_context = '\n'.join(
    catalog_summary(name) for name in ['flights.json', 'hotels.json', 'cars.json']
)

GROUNDED_INSTRUCTIONS = f"""
You are Contoso's enterprise travel concierge. Help with flights, hotels, and rental cars only.
Follow the corporate travel policy exactly, nudge non-compliant requests to compliant alternatives,
and ask for any missing booking details. When you mention options, prefer records that fit these
catalog summaries.

POLICY:
{policy_text}

CATALOG SUMMARIES:
{catalog_context}
""".strip()

print(GROUNDED_INSTRUCTIONS[:1000] + '\n... (truncated)')

## 2. Build the grounded agent

Same factory as round 1, but with the grounded instruction set instead of the generic one. Notice that the agent still has **no tools** — the only thing that changed is the system prompt.

In [ ]:
import asyncio, threading
from agents import make_chat_client, build_baseline_agent

client = make_chat_client()
agent = build_baseline_agent(client, GROUNDED_INSTRUCTIONS, name='travel_concierge_grounded')

# A single background thread owns one event loop. Both the kernel main
# thread (sanity call) and promptflow's worker threads (evaluate runs) hand
# their coroutines to this loop via run_coroutine_threadsafe, so MAF's
# telemetry ContextVar tokens are always set and reset inside the same task
# context.
_runner_loop = asyncio.new_event_loop()
_runner_thread = threading.Thread(
    target=_runner_loop.run_forever, name='eval-loop', daemon=True
)
_runner_thread.start()

async def _invoke(query: str):
    return await agent.run(query)

def target(query: str, **kwargs: object) -> dict:
    """Run the MAF agent for one dataset row and return the response.

    Per-row failures are swallowed so a single bad row never fails the whole
    batch and blocks evaluators from running on the rest of the dataset.
    """
    try:
        fut = asyncio.run_coroutine_threadsafe(_invoke(query), _runner_loop)
        response = fut.result()
        return {'response': response.text or '', 'query': query, 'error': ''}
    except Exception as exc:
        return {
            'response': f'[agent call failed: {type(exc).__name__}: {exc}]',
            'query': query,
            'error': f'{type(exc).__name__}: {exc}',
        }

# Quick sanity call before the long evaluate() run.
print(target('Find an economy flight from SFO to NYC on 2026-07-10 returning 2026-07-13 under $400.')['response'][:400])

## 3. Reuse the same evaluators

A fair comparison reuses the same evaluator suite. If you change the scoreboard between rounds, you can no longer tell whether the agent improved or whether the judges just got friendlier.

In [ ]:
from azure.identity import DefaultAzureCredential

# Cohere `command-a` is reachable on the account-level OpenAI-compatible path
# (/openai/v1/chat/completions), NOT on the legacy Azure OpenAI deployment path
# (/openai/deployments/{name}/chat/completions). So the evaluators' judge model
# must be configured as `type: openai` with base_url pointing at /openai/v1.
# We pass an Entra access token as the api_key (valid ~1 hour, well over the
# few minutes an eval run takes).
credential = DefaultAzureCredential()
_judge_token = credential.get_token('https://cognitiveservices.azure.com/.default').token
model_config = {
    'type': 'openai',
    'base_url': os.environ['AZURE_AI_ENDPOINT'].rstrip('/') + '/openai/v1',
    'model': DEPLOYMENT,
    'api_key': _judge_token,
}

def build_evaluators():
    import azure.ai.evaluation as evaluation
    evaluators = {}
    for name in ['RelevanceEvaluator', 'CoherenceEvaluator', 'FluencyEvaluator', 'GroundednessEvaluator']:
        cls = getattr(evaluation, name, None)
        if cls is None:
            continue
        try:
            evaluators[name.replace('Evaluator', '').lower()] = cls(model_config)
        except Exception as exc:
            # Preview versions sometimes change the config schema; skip rather than break the run.
            print(f'Skipping {name}: {exc}')
    # ToolCallAccuracyEvaluator is intentionally skipped: it requires
    # `tool_definitions` and `tool_calls` columns that the local target does
    # not emit. Add it back once those are captured per row.
    for name in ['IntentResolutionEvaluator', 'TaskAdherenceEvaluator']:
        cls = getattr(evaluation, name, None)
        if cls is None:
            continue
        try:
            evaluators[name.replace('Evaluator', '').lower()] = cls(model_config)
        except Exception as exc:
            print(f'Skipping {name}: {exc}')
    return evaluators

evaluators = build_evaluators()
print('Active evaluators:', sorted(evaluators))

## 4. Run the grounded evaluation

In [ ]:
from datetime import datetime
from azure.ai.evaluation import evaluate
import json as _json
import pandas as pd

# Cap parallel target + evaluator calls so the 100-RPM command-a deployment
# is not bursted past its rate limit. With 6 judge evaluators per row, even
# 4 workers will trigger 429s; 2 keeps the burst comfortably under 100 RPM.
# Set this BEFORE evaluate() builds its batch engine. Use plain assignment
# (not setdefault) so re-running the cell without a kernel restart picks up
# the new value if you ever tune it.
os.environ['PF_WORKER_COUNT'] = '2'

# Hub-less Microsoft Foundry projects are identified by their project endpoint;
# the legacy AzureAIProject(subscription, resource_group, project_name) triple
# resolves to an ML workspace that does not exist for these projects.
azure_ai_project = PROJECT_ENDPOINT

RESULTS_DIR = LAB_DIR / 'eval-results'
RESULTS_DIR.mkdir(exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
output_path = RESULTS_DIR / f'round2-grounded-{timestamp}.json'

def _run_evaluation(project):
    return evaluate(
        data=str(DATASET),
        target=target,
        evaluators=evaluators,
        azure_ai_project=project,
        evaluation_name='round2-grounded',
        fail_on_evaluator_errors=False,
        tags={'lab': 'cohere-lab-1-foundry-maf', 'round': 'round2-grounded'},
        output_path=str(output_path),
    )

try:
    result = _run_evaluation(azure_ai_project)
    uploaded = True
except Exception as exc:
    print(f'Portal upload failed ({type(exc).__name__}: {exc}); rerunning local-only.')
    result = _run_evaluation(None)
    uploaded = False

metrics = getattr(result, 'metrics', None) or (result.get('metrics') if isinstance(result, dict) else {})
rows = getattr(result, 'rows', None) or (result.get('rows') if isinstance(result, dict) else [])

rows_df = pd.DataFrame(rows)
rows_path = RESULTS_DIR / f'round2-grounded-{timestamp}-rows.csv'
metrics_path = RESULTS_DIR / f'round2-grounded-{timestamp}-metrics.json'
rows_df.to_csv(rows_path, index=False)
metrics_path.write_text(_json.dumps(metrics, indent=2, sort_keys=True))

print(f'Portal upload: {"yes" if uploaded else "no (local-only)"}')
print(f'Full result : {output_path}')
print(f'Per-row CSV : {rows_path}')
print(f'Metrics JSON: {metrics_path}')
pd.DataFrame(sorted(metrics.items()), columns=['metric', 'value'])

## What you confirmed

- Grounding the agent with policy + catalog facts is a one-line change to the instructions.
- You can now compare `round2-grounded-*-metrics.json` to `round1-baseline-*-metrics.json` side by side.
- Built-in evaluators capture *general* quality (relevance, coherence, groundedness). They do not, however, know about *your* policy specifics.

Next: **`04-eval-round3-policy-evaluator.ipynb`** adds a custom corporate-travel auditor to the evaluator panel.